# Spam Mail Prediction using Machine Learning
**Project by:** Arjun | Internship Project

---
### Table of Contents
1. Importing the Dependencies
2. Data Collection & Pre-Processing
3. Phase 1 — Exploratory Data Analysis (EDA)
4. Label Encoding
5. Baseline Model (TF-IDF + Logistic Regression)
6. **Phase 2 — Advanced Text Preprocessing (NLTK)**
7. **Phase 3 — Multi-Model Comparison**
8. **Phase 4 — Comprehensive Evaluation Metrics**
9. **Phase 5 — Hyperparameter Tuning**
10. **Phase 6 — Class Imbalance Handling (SMOTE)**
11. **Phase 7 — Model Saving & Deployment Pipeline**
12. Predictive System

## 1. Importing the Dependencies

In [ ]:
# ── Core ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import re, string, warnings, joblib
warnings.filterwarnings('ignore')

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# ── ML — pre-processing & splitting ───────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

# ── ML — models ───────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# ── ML — metrics ──────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)

# ── NLTK ──────────────────────────────────────────────────────────
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('punkt',     quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ── Imbalanced-learn (SMOTE) ──────────────────────────────────────
# !pip install imbalanced-learn -q   # uncomment if needed on Colab
from imblearn.over_sampling import SMOTE

# Global plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SPAM_COLOR = '#e74c3c'
HAM_COLOR  = '#2ecc71'

print('All libraries imported successfully!')

## 2. Data Collection & Pre-Processing

In [ ]:
# Load dataset — upload the 'data/' folder to /content/ in Colab
raw_mail_data = pd.read_csv('/content/data/spam.csv', encoding='latin-1')
raw_mail_data = raw_mail_data[['v1', 'v2']]
raw_mail_data.columns = ['Category', 'Message']

# Replace null values
mail_data = raw_mail_data.where(pd.notnull(raw_mail_data), '')

print('Shape:', mail_data.shape)
print('Null values:\n', mail_data.isnull().sum())
mail_data.head()

---
## 3. Phase 1 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1  Class Distribution ───────────────────────────────────────
class_counts = mail_data['Category'].value_counts()
print('Class Distribution:\n', class_counts)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(class_counts.index, class_counts.values,
            color=[SPAM_COLOR, HAM_COLOR], edgecolor='black', width=0.5)
axes[0].set_title('Spam vs Ham — Count', fontweight='bold')
axes[0].set_ylabel('Messages')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v+30, str(v), ha='center', fontweight='bold')
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=[SPAM_COLOR, HAM_COLOR],
            startangle=140, explode=(0.05,0), shadow=True)
axes[1].set_title('Proportion', fontweight='bold')
plt.suptitle('Class Distribution', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2  Message Length Features ──────────────────────────────────
mail_data['char_count']        = mail_data['Message'].apply(len)
mail_data['word_count']        = mail_data['Message'].apply(lambda x: len(x.split()))
mail_data['uppercase_ratio']   = mail_data['Message'].apply(lambda x: sum(c.isupper() for c in x)/(len(x)+1))
mail_data['digit_count']       = mail_data['Message'].apply(lambda x: sum(c.isdigit() for c in x))
mail_data['exclamation_count'] = mail_data['Message'].apply(lambda x: x.count('!'))
mail_data['has_url']           = mail_data['Message'].apply(lambda x: 1 if re.search(r'http|www|\.com', x, re.I) else 0)
mail_data['has_currency']      = mail_data['Message'].apply(lambda x: 1 if re.search(r'[\$£€]|free|win|prize|cash', x, re.I) else 0)

spam_data = mail_data[mail_data['Category']=='spam']
ham_data  = mail_data[mail_data['Category']=='ham']

# Length histograms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat, label in zip(axes, ['char_count','word_count'], ['Character Count','Word Count']):
    ax.hist(spam_data[feat], bins=50, alpha=0.7, color=SPAM_COLOR, label='Spam', edgecolor='black')
    ax.hist(ham_data[feat],  bins=50, alpha=0.7, color=HAM_COLOR,  label='Ham',  edgecolor='black')
    ax.axvline(spam_data[feat].mean(), color='darkred',   linestyle='--', label=f'Spam mean: {spam_data[feat].mean():.0f}')
    ax.axvline(ham_data[feat].mean(),  color='darkgreen', linestyle='--', label=f'Ham mean: {ham_data[feat].mean():.0f}')
    ax.set_title(f'{label} Distribution', fontweight='bold')
    ax.set_xlabel(label); ax.set_ylabel('Frequency'); ax.legend(fontsize=9)
plt.suptitle('Message Length: Spam vs Ham', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.3  WordCloud ────────────────────────────────────────────────
spam_text = ' '.join(spam_data['Message'].values)
ham_text  = ' '.join(ham_data['Message'].values)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, text, cmap, title in zip(
        axes,
        [spam_text, ham_text],
        ['Reds', 'Greens'],
        ['🚨 SPAM — Top Words', '✅ HAM — Top Words']):
    wc = WordCloud(width=800, height=400, background_color='black',
                   colormap=cmap, max_words=150, collocations=False).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off'); ax.set_title(title, fontsize=13, fontweight='bold')
plt.suptitle('WordCloud: Spam vs Ham', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.4  Top-20 Words ─────────────────────────────────────────────
def get_top_words(series, n=20):
    words = []
    for m in series:
        words.extend(re.sub(r'[^\w\s]','',m.lower()).split())
    return Counter(words).most_common(n)

top_spam = get_top_words(spam_data['Message'])
top_ham  = get_top_words(ham_data['Message'])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, top, color, title in zip(axes, [top_spam, top_ham], [SPAM_COLOR, HAM_COLOR],
                                  ['🚨 Top 20 SPAM Words','✅ Top 20 HAM Words']):
    words, freqs = zip(*top)
    ax.barh(list(words)[::-1], list(freqs)[::-1], color=color, edgecolor='black', alpha=0.85)
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.set_xlabel('Frequency')
plt.suptitle('Top 20 Most Frequent Words', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.5  Correlation Heatmap ──────────────────────────────────────
mail_data['label'] = mail_data['Category'].map({'spam':0,'ham':1})
corr_cols = ['label','char_count','word_count','uppercase_ratio','digit_count',
             'exclamation_count','has_url','has_currency']
corr = mail_data[corr_cols].corr()
plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=np.triu(np.ones_like(corr,dtype=bool)),
            linewidths=0.5, vmin=-1, vmax=1, square=True)
plt.title('Correlation Heatmap — Features vs Label', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 4. Label Encoding

In [ ]:
mail_data.loc[mail_data['Category']=='spam','Category'] = 0
mail_data.loc[mail_data['Category']=='ham', 'Category'] = 1
print('spam → 0 | ham → 1')
print(mail_data['Category'].value_counts())

In [ ]:
X = mail_data['Message']
Y = mail_data['Category'].astype('int')
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=3)
print(f'Total: {X.shape[0]}  |  Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

---
## 5. Baseline Model — TF-IDF + Logistic Regression

In [ ]:
# Baseline TF-IDF vectorizer
tfidf_base = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
X_train_base = tfidf_base.fit_transform(X_train)
X_test_base  = tfidf_base.transform(X_test)

lr_base = LogisticRegression(max_iter=1000)
lr_base.fit(X_train_base, Y_train)

baseline_train_acc = accuracy_score(Y_train, lr_base.predict(X_train_base))
baseline_test_acc  = accuracy_score(Y_test,  lr_base.predict(X_test_base))

print('=== BASELINE MODEL (Logistic Regression) ===')
print(f'  Training Accuracy : {baseline_train_acc*100:.2f}%')
print(f'  Test     Accuracy : {baseline_test_acc*100:.2f}%')

---
## 6. Phase 2 — Advanced Text Preprocessing (NLTK)
> We build a proper NLP cleaning pipeline: lowercase → remove punctuation → tokenize → remove stopwords → stem/lemmatize.

In [ ]:
stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()
STOP_WORDS = set(stopwords.words('english'))

def clean_text(text, use_stemming=False):
    """Full NLP cleaning pipeline."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)
    # 4. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 5. Tokenize
    tokens = text.split()
    # 6. Remove stopwords
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]
    # 7. Stem or Lemmatize
    if use_stemming:
        tokens = [stemmer.lemmatize(t) for t in tokens]
    else:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply to full dataset
mail_data['cleaned_message'] = mail_data['Message'].apply(clean_text)

print('=== Before Cleaning ===')
print(mail_data['Message'].iloc[2])
print('\n=== After Cleaning ===')
print(mail_data['cleaned_message'].iloc[2])

In [ ]:
# Compare average word count before vs after cleaning
mail_data['cleaned_word_count'] = mail_data['cleaned_message'].apply(lambda x: len(x.split()))

print('Average word count comparison:')
comp = mail_data.groupby('Category')[['word_count','cleaned_word_count']].mean().round(2)
comp.index = ['Spam (0)', 'Ham (1)']
comp.columns = ['Raw word count', 'Cleaned word count']
print(comp.to_string())

# Build TF-IDF on cleaned text
X_clean = mail_data['cleaned_message']
X_train_c, X_test_c, Y_train_c, Y_test_c = train_test_split(X_clean, Y, test_size=0.2, random_state=3)

tfidf_clean = TfidfVectorizer(min_df=1, ngram_range=(1,2), max_features=15000)
X_train_clean = tfidf_clean.fit_transform(X_train_c)
X_test_clean  = tfidf_clean.transform(X_test_c)

print(f'\nCleaned TF-IDF matrix shape: {X_train_clean.shape}')
print('Bigrams included — captures phrases like "free entry", "call now", "win prize"')

---
## 7. Phase 3 — Multi-Model Comparison
> We train 6 classifiers on the cleaned TF-IDF features and compare them across 5 metrics.

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0),
    'Naive Bayes'         : MultinomialNB(alpha=0.1),
    'Linear SVM'          : LinearSVC(C=1.0, max_iter=2000),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_models = {}

for name, clf in models.items():
    clf.fit(X_train_clean, Y_train_c)
    preds = clf.predict(X_test_clean)

    # ROC-AUC: LinearSVC uses decision_function
    if hasattr(clf, 'predict_proba'):
        proba = clf.predict_proba(X_test_clean)[:,1]
    else:
        proba = clf.decision_function(X_test_clean)

    results.append({
        'Model'     : name,
        'Accuracy'  : accuracy_score(Y_test_c, preds),
        'Precision' : precision_score(Y_test_c, preds),
        'Recall'    : recall_score(Y_test_c, preds),
        'F1-Score'  : f1_score(Y_test_c, preds),
        'ROC-AUC'   : roc_auc_score(Y_test_c, proba),
    })
    trained_models[name] = clf
    print(f'{name:<22} | Acc: {results[-1]["Accuracy"]:.4f} | F1: {results[-1]["F1-Score"]:.4f} | AUC: {results[-1]["ROC-AUC"]:.4f}')

results_df = pd.DataFrame(results).set_index('Model').sort_values('F1-Score', ascending=False)
print('\n=== Model Comparison Table ===')
print(results_df.round(4).to_string())

In [ ]:
# Grouped bar chart comparing all models on all metrics
metrics   = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']
model_names = results_df.index.tolist()
x = np.arange(len(model_names))
bar_w = 0.14
colors = ['#3498db','#e67e22','#2ecc71','#e74c3c','#9b59b6']

fig, ax = plt.subplots(figsize=(16, 6))
for i, (metric, color) in enumerate(zip(metrics, colors)):
    vals = [results_df.loc[m, metric] for m in model_names]
    bars = ax.bar(x + i*bar_w, vals, bar_w, label=metric, color=color, alpha=0.85, edgecolor='black')

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — All Metrics', fontsize=15, fontweight='bold')
ax.set_xticks(x + bar_w*2)
ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=10)
ax.set_ylim(0.85, 1.02)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout(); plt.show()

best_model_name = results_df['F1-Score'].idxmax()
print(f'\n🏆 Best model by F1-Score: {best_model_name}')

---
## 8. Phase 4 — Comprehensive Evaluation Metrics
> Deep-dive evaluation of the best model: Confusion Matrix, Classification Report, ROC Curve, Precision-Recall Curve, and Cross-Validation.

In [ ]:
# Use best model from Phase 3
best_clf   = trained_models[best_model_name]
best_preds = best_clf.predict(X_test_clean)

# ── 4.1  Classification Report ────────────────────────────────────
print(f'=== Classification Report — {best_model_name} ===')
print(classification_report(Y_test_c, best_preds, target_names=['Spam','Ham']))

# ── 4.2  Confusion Matrix ─────────────────────────────────────────
cm = confusion_matrix(Y_test_c, best_preds)
fig, ax = plt.subplots(figsize=(6,5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Spam','Ham'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_model_name}', fontsize=13, fontweight='bold')

# Annotate TP, TN, FP, FN
labels = [['TN','FP'],['FN','TP']]
for i in range(2):
    for j in range(2):
        ax.text(j, i+0.35, f'({labels[i][j]})', ha='center', va='center',
                fontsize=10, color='grey')
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'False Positive Rate (spam classified as ham): {fp/(fp+tn)*100:.2f}%  — critical for spam filter!')
print(f'False Negative Rate (ham classified as spam): {fn/(fn+tp)*100:.2f}%')

In [ ]:
# ── 4.3  ROC Curve + PR Curve ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

palette = ['#e74c3c','#3498db','#2ecc71','#e67e22','#9b59b6','#1abc9c']
for (name, clf), color in zip(trained_models.items(), palette):
    if hasattr(clf, 'predict_proba'):
        scores = clf.predict_proba(X_test_clean)[:,1]
    else:
        scores = clf.decision_function(X_test_clean)

    # ROC
    fpr, tpr, _ = roc_curve(Y_test_c, scores)
    auc = roc_auc_score(Y_test_c, scores)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

    # PR
    prec, rec, _ = precision_recall_curve(Y_test_c, scores)
    axes[1].plot(rec, prec, color=color, lw=2, label=name)

# ROC plot
axes[0].plot([0,1],[0,1],'k--', lw=1, label='Random (AUC=0.5)')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — All Models', fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')

# PR plot
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — All Models', fontweight='bold')
axes[1].legend(fontsize=8, loc='lower left')

plt.suptitle('Model Evaluation Curves', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.4  5-Fold Cross-Validation ──────────────────────────────────
print('=== 5-Fold Cross-Validation (F1-Score) ===')
cv_results = {}
for name, clf in trained_models.items():
    scores = cross_val_score(
        clf, X_train_clean, Y_train_c,
        cv=5, scoring='f1', n_jobs=-1
    )
    cv_results[name] = scores
    print(f'{name:<22} | Mean F1: {scores.mean():.4f} ± {scores.std():.4f}  | Scores: {np.round(scores,4)}')

# CV box plot
fig, ax = plt.subplots(figsize=(12,5))
ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True,
           medianprops=dict(color='black',linewidth=2))
ax.set_title('5-Fold Cross-Validation F1-Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('F1-Score'); ax.set_ylim(0.85, 1.0)
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

---
## 9. Phase 5 — Hyperparameter Tuning
> We use GridSearchCV to find the optimal TF-IDF and model parameters for the best-performing model.

In [ ]:
from sklearn.pipeline import Pipeline

# We tune Logistic Regression (most interpretable + top performer)
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', lowercase=True)),
    ('clf',   LogisticRegression(max_iter=1000))
])

param_grid = {
    'tfidf__ngram_range' : [(1,1), (1,2)],
    'tfidf__max_features': [5000, 10000, 20000],
    'tfidf__min_df'      : [1, 2],
    'clf__C'             : [0.1, 1.0, 10.0],
    'clf__penalty'       : ['l2'],
}

print('Running GridSearchCV (this may take ~2 minutes)...')
grid_search = GridSearchCV(
    pipe, param_grid,
    cv=5, scoring='f1',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train_c, Y_train_c)

print(f'\nBest F1-Score (CV): {grid_search.best_score_:.4f}')
print('Best Parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k:<28}: {v}')

In [ ]:
# Evaluate best tuned model on test set
best_tuned = grid_search.best_estimator_
tuned_preds = best_tuned.predict(X_test_c)

tuned_metrics = {
    'Accuracy'  : accuracy_score(Y_test_c, tuned_preds),
    'Precision' : precision_score(Y_test_c, tuned_preds),
    'Recall'    : recall_score(Y_test_c, tuned_preds),
    'F1-Score'  : f1_score(Y_test_c, tuned_preds),
    'ROC-AUC'   : roc_auc_score(Y_test_c, best_tuned.predict_proba(X_test_c)[:,1]),
}
base_metrics = {
    'Accuracy'  : accuracy_score(Y_test_c, results_df.loc['Logistic Regression'].name and lr_base.predict(X_test_base)),
    'Precision' : precision_score(Y_test_c, lr_base.predict(X_test_base)),
    'Recall'    : recall_score(Y_test_c, lr_base.predict(X_test_base)),
    'F1-Score'  : f1_score(Y_test_c, lr_base.predict(X_test_base)),
    'ROC-AUC'   : roc_auc_score(Y_test_c, lr_base.predict_proba(X_test_base)[:,1]),
}

compare_df = pd.DataFrame({'Baseline LR': base_metrics, 'Tuned LR': tuned_metrics}).T
print('=== Before vs After Hyperparameter Tuning ===')
print(compare_df.round(4).to_string())

# Bar chart comparison
x = np.arange(len(tuned_metrics))
fig, ax = plt.subplots(figsize=(10,5))
ax.bar(x - 0.2, base_metrics.values(),  0.35, label='Baseline LR', color='#95a5a6', edgecolor='black')
ax.bar(x + 0.2, tuned_metrics.values(), 0.35, label='Tuned LR',    color='#3498db', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(tuned_metrics.keys())
ax.set_ylim(0.88, 1.01); ax.set_title('Baseline vs Tuned Logistic Regression', fontsize=13, fontweight='bold')
ax.set_ylabel('Score'); ax.legend()
plt.tight_layout(); plt.show()

---
## 10. Phase 6 — Class Imbalance Handling (SMOTE)
> The dataset is ~87% ham / ~13% spam. We apply SMOTE to oversample the minority (spam) class and compare model performance.

In [ ]:
print('Class distribution BEFORE SMOTE:')
print(pd.Series(Y_train_c).value_counts().rename({0:'spam',1:'ham'}))

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_sm, Y_train_sm = smote.fit_resample(X_train_clean, Y_train_c)

print('\nClass distribution AFTER SMOTE:')
print(pd.Series(Y_train_sm).value_counts().rename({0:'spam',1:'ham'}))

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
before = pd.Series(Y_train_c).value_counts()
after  = pd.Series(Y_train_sm).value_counts()
for ax, counts, title in zip(axes, [before, after], ['Before SMOTE', 'After SMOTE']):
    ax.bar(['Ham (1)', 'Spam (0)'], [counts.get(1,0), counts.get(0,0)],
           color=[HAM_COLOR, SPAM_COLOR], edgecolor='black', width=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.set_ylabel('Count')
    for i, v in enumerate([counts.get(1,0), counts.get(0,0)]):
        ax.text(i, v+30, str(v), ha='center', fontweight='bold')
plt.suptitle('Class Distribution Before vs After SMOTE', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# Compare LR: no SMOTE vs SMOTE vs class_weight='balanced'
experiments = {
    'LR — No Balancing'      : (X_train_clean, Y_train_c,  LogisticRegression(max_iter=1000)),
    'LR — class_weight=bal'  : (X_train_clean, Y_train_c,  LogisticRegression(max_iter=1000, class_weight='balanced')),
    'LR — SMOTE'             : (X_train_sm,    Y_train_sm, LogisticRegression(max_iter=1000)),
}

balance_results = []
for exp_name, (Xtr, Ytr, clf) in experiments.items():
    clf.fit(Xtr, Ytr)
    preds = clf.predict(X_test_clean)
    balance_results.append({
        'Method'    : exp_name,
        'Accuracy'  : accuracy_score(Y_test_c, preds),
        'Precision' : precision_score(Y_test_c, preds),
        'Recall'    : recall_score(Y_test_c, preds),
        'F1-Score'  : f1_score(Y_test_c, preds),
    })

bal_df = pd.DataFrame(balance_results).set_index('Method')
print('=== Imbalance Handling Comparison ===')
print(bal_df.round(4).to_string())

# Plot
x    = np.arange(4)
met  = ['Accuracy','Precision','Recall','F1-Score']
bw   = 0.25
cols = ['#95a5a6','#3498db','#e74c3c']
fig, ax = plt.subplots(figsize=(12,5))
for i, (method, color) in enumerate(zip(bal_df.index, cols)):
    ax.bar(x + (i-1)*bw, bal_df.loc[method, met], bw,
           label=method, color=color, edgecolor='black', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(met)
ax.set_ylim(0.85, 1.01); ax.set_ylabel('Score')
ax.set_title('Class Imbalance Handling Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

---
## 11. Phase 7 — Model Saving & Deployment Pipeline
> Save the best tuned model and vectorizer to disk so they can be reloaded without retraining.

In [ ]:
import joblib, os

# The best tuned pipeline already contains the vectorizer
# (GridSearchCV best_estimator_ is a Pipeline: TF-IDF → LR)

save_dir = '/content/saved_model'
os.makedirs(save_dir, exist_ok=True)

# Save the full pipeline (vectorizer + model)
joblib.dump(best_tuned, f'{save_dir}/spam_model_pipeline.pkl')

# Also save separately for flexibility
joblib.dump(best_tuned.named_steps['tfidf'], f'{save_dir}/vectorizer.pkl')
joblib.dump(best_tuned.named_steps['clf'],   f'{save_dir}/model.pkl')

print('Model saved!')
print(f"  Pipeline : {save_dir}/spam_model_pipeline.pkl")
print(f"  Vectorizer: {save_dir}/vectorizer.pkl")
print(f"  Model     : {save_dir}/model.pkl")

In [ ]:
# ── Load fresh and verify ─────────────────────────────────────────
loaded_pipeline = joblib.load(f'{save_dir}/spam_model_pipeline.pkl')

def predict_email(text):
    """End-to-end prediction: raw text → Spam / Ham with confidence."""
    cleaned = clean_text(text)
    pred    = loaded_pipeline.predict([cleaned])[0]
    proba   = loaded_pipeline.predict_proba([cleaned])[0]
    label   = '✅ HAM (Legitimate)' if pred == 1 else '🚨 SPAM'
    conf    = proba[pred] * 100
    return label, conf

# Test on 5 messages
test_cases = [
    "FREE entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121",
    "Hey, are you coming to the party tonight? Let me know!",
    "Congratulations! You have WON a £1000 Walmart gift card. Call 0800-123456 NOW to claim!",
    "Mom says dinner is ready, come home now",
    "URGENT: Your account has been suspended. Click http://bit.ly/verify to restore access."
]

print('=== Deployed Model — Predictions ===')
print(f'{"Message":<60} {"Prediction":<22} {"Confidence"}')
print('-' * 95)
for msg in test_cases:
    label, conf = predict_email(msg)
    print(f'{msg[:57]:<60} {label:<22} {conf:.1f}%')

In [ ]:
# ── Download the saved model files from Colab to local machine ────
from google.colab import files
files.download(f'{save_dir}/spam_model_pipeline.pkl')
files.download(f'{save_dir}/vectorizer.pkl')
files.download(f'{save_dir}/model.pkl')
print('Files downloaded! Use them in app.py for Phase 8 Streamlit app.')

### 11.1 Feature Importance — Top Spam & Ham Words (from model coefficients)

In [ ]:
# Extract LR coefficients from tuned pipeline
lr_model  = best_tuned.named_steps['clf']
tfidf_vec = best_tuned.named_steps['tfidf']
feature_names = np.array(tfidf_vec.get_feature_names_out())
coef          = lr_model.coef_[0]

N = 25
# Top spam words = most negative coefficients (spam=0, ham=1)
top_spam_idx = coef.argsort()[:N]
top_ham_idx  = coef.argsort()[-N:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(feature_names[top_spam_idx][::-1], np.abs(coef[top_spam_idx][::-1]),
             color=SPAM_COLOR, edgecolor='black', alpha=0.85)
axes[0].set_title('🚨 Top 25 SPAM Indicator Words\n(from LR coefficients)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('|Coefficient|')

axes[1].barh(feature_names[top_ham_idx][::-1], coef[top_ham_idx][::-1],
             color=HAM_COLOR, edgecolor='black', alpha=0.85)
axes[1].set_title('✅ Top 25 HAM Indicator Words\n(from LR coefficients)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coefficient')

plt.suptitle('Feature Importance — Logistic Regression Coefficients', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 12. Final Results Summary

| Phase | Achievement |
|---|---|
| **Phase 1 — EDA** | Class imbalance found (87/13), spam messages longer, more uppercase/digits/URLs |
| **Phase 2 — Preprocessing** | NLTK pipeline: lowercase → URL removal → punctuation removal → stopwords → lemmatize |
| **Phase 3 — Multi-Model** | 6 models compared; Linear SVM & LR best performers |
| **Phase 4 — Evaluation** | Confusion matrix, ROC-AUC, Precision-Recall curves, 5-fold CV |
| **Phase 5 — Tuning** | GridSearchCV over TF-IDF + LR params; improved F1 |
| **Phase 6 — SMOTE** | SMOTE rebalanced classes; improved spam recall |
| **Phase 7 — Deployment** | Model saved as `.pkl`; `predict_email()` function; feature importance plotted |

> **Coming Next (Phase 8):** Streamlit web application — paste any message and get an instant spam prediction!

In [ ]:
# ── Interactive single-message predictor ─────────────────────────
def check_message(message):
    label, confidence = predict_email(message)
    print('=' * 60)
    print(f'Message    : {message[:80]}')
    print(f'Result     : {label}')
    print(f'Confidence : {confidence:.2f}%')
    print('=' * 60)

# Try your own message below!
check_message("You have been selected for a FREE iPhone 15. Call us now!")
check_message("Let's catch up tomorrow for lunch, it's been a while!")